In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
import numpy as np

c:\Users\тема\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
input_file = r"C:\Users\тема\Desktop\Реплики.csv"
output_file = r"C:\Users\тема\Desktop\Реплики_clean.csv"

In [3]:
text_col = "Реплика"
label_col = "Эмоция"
to_drop = ["Neutral", "Неграмматично"]

emotion_map = {
    "Joy": 0,
    "Sadness": 1,
    "Surprise": 2,
    "Fear": 3,
    "Anger": 4
}

In [6]:
df = pd.read_csv(input_file, sep=';')

df = df[~df[label_col].isin(to_drop)]

df["label"] = df[label_col].map(emotion_map)

df = df.dropna(subset=["label"])

df = df[[text_col, "label"]].rename(columns={text_col: "text"})
df["label"] = df["label"].astype(int)

df.to_csv(output_file, index=False)
print(f"Сохранено {len(df)} строк в {output_file}")

Сохранено 5134 строк в C:\Users\тема\Desktop\Реплики_clean.csv


In [11]:
model_path = r"C:\Users\тема\Desktop\saved_model_rubert-tiny2"
data_path = r"C:\Users\тема\Desktop\Реплики_clean.csv"
emotions = ["joy", "sadness", "surprise", "fear", "anger"]
max_len = 100  # как при обучении

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

df_eval = pd.read_csv(data_path)
texts = df_eval["text"].fillna("").astype(str).tolist()
true_labels = df_eval["label"].values

all_preds = []
with torch.no_grad():
    for i in tqdm(range(0, len(texts), 32), desc="rubert-tiny2"):
        batch = texts[i:i+32]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        ).to(device)

        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)

all_preds = np.array(all_preds)

print("\n" + "="*50)
print("rubert-tiny2")
print("="*50)
print(classification_report(true_labels, all_preds, target_names=emotions, digits=3))


rubert-tiny2: 100%|██████████| 161/161 [00:01<00:00, 96.11it/s]


rubert-tiny2
              precision    recall  f1-score   support

         joy      0.424     0.230     0.298      1290
     sadness      0.370     0.234     0.287      1191
    surprise      0.497     0.492     0.494      1130
        fear      0.427     0.190     0.263       584
       anger      0.284     0.696     0.404       939

    accuracy                          0.369      5134
   macro avg      0.400     0.369     0.349      5134
weighted avg      0.402     0.369     0.354      5134



In [12]:
model_path = r"C:\Users\тема\Desktop\saved_model_rubert-base-cased"
data_path = r"C:\Users\тема\Desktop\Реплики_clean.csv"
emotions = ["joy", "sadness", "surprise", "fear", "anger"]
max_len = 100
batch_size = 16

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

df_eval = pd.read_csv(data_path)
texts = df_eval["text"].fillna("").astype(str).tolist()
true_labels = df_eval["label"].values


all_preds = []
with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="rubert-base-cased"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        ).to(device)

        preds = torch.argmax(model(**inputs).logits, dim=1).cpu().numpy()
        all_preds.extend(preds)

all_preds = np.array(all_preds)

print("\n" + "="*50)
print("DeepPavlov/rubert-base-cased")
print("="*50)
print(classification_report(true_labels, all_preds, target_names=emotions, digits=3))

rubert-base-cased: 100%|██████████| 321/321 [00:34<00:00,  9.39it/s]


DeepPavlov/rubert-base-cased
              precision    recall  f1-score   support

         joy      0.451     0.291     0.354      1290
     sadness      0.436     0.226     0.298      1191
    surprise      0.521     0.690     0.594      1130
        fear      0.499     0.288     0.365       584
       anger      0.299     0.590     0.397       939

    accuracy                          0.418      5134
   macro avg      0.441     0.417     0.402      5134
weighted avg      0.441     0.418     0.403      5134

